In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import numpy as np

In [ ]:
# Check for GPU availability
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if torch.cuda.is_available():
    print(f"Used GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU found, using CPU!")

In [ ]:
print("\nLoading data from .npy files...")
X_train_np = np.load('../data/processed/X_train_resnet.npy')
y_train_np = np.load('../data/processed/y_train.npy')

X_test_np = np.load('../data/processed/X_test_resnet.npy')
y_test_np = np.load('../data/processed/y_test.npy')

X_train = torch.tensor(X_train_np, dtype=torch.float32)
y_train = torch.tensor(y_train_np, dtype=torch.long)

X_test = torch.tensor(X_test_np, dtype=torch.float32)
y_test = torch.tensor(y_test_np, dtype=torch.long)

batch_size = 128
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True) 

test_dataset = TensorDataset(X_test, y_test)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
print("DataLoaders are ready.")

In [ ]:
# Define Neural Network (MLP) Class
class SkinLesionMLP(nn.Module):
    def __init__(self):
        super(SkinLesionMLP, self).__init__()
        
        # Layer 1: 2048 features -> 512 hidden units [cite: 19, 31]
        self.fc1 = nn.Linear(2048, 512)
        self.relu1 = nn.ReLU() # Non-linear activation
        self.dropout1 = nn.Dropout(p=0.5) # Prevent overfitting
        
        # Layer 2: 512 -> 128 hidden units
        self.fc2 = nn.Linear(512, 128)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(p=0.5)
        
        # Output Layer: 128 -> 7 Diagnostic Classes [cite: 21, 48]
        self.fc3 = nn.Linear(128, 7)

    def forward(self, x):
        # Forward pass sequence
        x = self.fc1(x)
        x = self.relu1(x)
        x = self.dropout1(x)
        
        x = self.fc2(x)
        x = self.relu2(x)
        x = self.dropout2(x)
        
        x = self.fc3(x) 
        # No Softmax here; handled by nn.CrossEntropyLoss [cite: 51]
        return x

# Initialize the model and move it to the device (GPU/CPU)
model = SkinLesionMLP().to(device)

print("Model Architecture Successfully Created:")
print(model)

In [ ]:
# Loss Function and Optimizer
criterion = nn.CrossEntropyLoss()

# Adam optimizer for efficient weight (W) updates [cite: 51]
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training Loop
num_epochs = 30 

print("Starting training\n")

for epoch in range(num_epochs):
    # A) TRAINING PHASE
    model.train() # Enable dropout [cite: 5]
    running_loss = 0.0
    
    for inputs, labels in train_loader:
        # Move data to GPU
        inputs, labels = inputs.to(device), labels.to(device)
        
        # Reset gradients
        optimizer.zero_grad()
        
        # Forward pass: compute predictions
        outputs = model(inputs)
        
        # Calculate loss [cite: 52]
        loss = criterion(outputs, labels)
        
        # Backward pass: compute gradients
        loss.backward()
        
        # Update weights
        optimizer.step()
        
        running_loss += loss.item() * inputs.size(0)
        
    epoch_loss = running_loss / len(train_loader.dataset)
    
    # B) EVALUATION PHASE
    model.eval() # Disable dropout for inference
    correct = 0
    total = 0
    
    with torch.no_grad(): # No gradient computation needed
        for test_inputs, test_labels in test_loader:
            test_inputs, test_labels = test_inputs.to(device), test_labels.to(device)
            
            outputs = model(test_inputs)
            _, predicted = torch.max(outputs.data, 1) # Get predicted class
            
            total += test_labels.size(0)
            correct += (predicted == test_labels).sum().item()
            
    test_accuracy = 100 * correct / total
    
    # Print progress every 5 epochs
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch [{epoch+1:2d}/{num_epochs}] | Training Loss: {epoch_loss:.4f} | Test Accuracy: %{test_accuracy:.2f}")

print("\nTraining Complete!")
    

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

# Get predictions and true labels for the test set
model.eval()
all_preds = []
all_targets = []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(device)
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        
        all_preds.extend(preds.cpu().numpy())
        all_targets.extend(labels.numpy())

# 0=nv, 1=mel, 2=bkl, 3=bcc, 4=akiec, 5=vasc, 6=df
class_names = ['nv', 'mel', 'bkl', 'bcc', 'akiec', 'vasc', 'df']

# Report
print("--- CLASSIFICATION REPORT ---")
print(classification_report(all_targets, all_preds, target_names=class_names))

# Confusion Matrix
cm = confusion_matrix(all_targets, all_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=class_names, yticklabels=class_names)
plt.title('MLP Baseline - Confusion Matrix')
plt.ylabel('True Labels')
plt.xlabel('Predicted Labels')
plt.show()